# GPU Challenge 2026 - Scoring Alternatif Benin
**Cas d'usage 3 : Evaluation du risque de credit des petits commercants**

Notebook optimise pour **Google Colab GPU (T4/V100/A100)** - Exploitation massive du GPU

---

## Objectifs
1. **Feature Engineering avance** : Creation de 100+ features derivees
2. **XGBoost GPU** : Entrainement avec `tree_method='gpu_hist'`, `predictor='gpu_predictor'`
3. **SHAP GPU** : Explicabilite acceleree sur GPU
4. **Regles metier IMF** : Integration des plafonds, taux d'usure (27%), renouvellements
5. **Benchmark GPU vs CPU** : Demonstration de l'acceleration
6. **Hyperparameter Tuning GPU** : Optuna avec parallelisation GPU
7. **Ensemble Stacking GPU** : XGBoost + LightGBM + CatBoost sur GPU
8. **Deep Feature Synthesis** : Feature engineering automatise intensif

In [1]:
# ============================================================
# CELLULE 1: VERIFICATION GPU OBLIGATOIRE KAGGLE
# ============================================================

import os
import subprocess
import sys

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
os.environ['XGBOOST_USE_CUDA'] = '1'

def run_cmd(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    return result

print("="*60)
print("VERIFICATION GPU OBLIGATOIRE - KAGGLE GPU CHALLENGE")
print("="*60)

run_cmd("nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version,compute_cap --format=csv")
run_cmd("nvcc --version")

import torch
assert torch.cuda.is_available(), "ERREUR: Pas de GPU sur Kaggle. Activez GPU: Settings -> GPU -> On"
print(f"PyTorch CUDA: {torch.version.cuda}")
print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

import tensorflow as tf
gpu_devices = tf.config.list_physical_devices('GPU')
assert len(gpu_devices) > 0, "ERREUR: TensorFlow ne detecte pas de GPU"
print(f"TF GPU Devices: {gpu_devices}")

import xgboost as xgb
import cupy as cp
import numpy as np

print(f"XGBoost Version: {xgb.__version__}")

# Test reel: XGBoost 3.x - tree_method='hist' + device='cuda' pour GPU
# IL FAUT DES LABELS pour l'entrainement
try:
    test_arr = cp.array([[1.0, 2.0, 3.0]])  # 2D shape (1, 3)
    test_labels = cp.array([0.0])  # Label requis pour binaire
    dtrain_test = xgb.DMatrix(test_arr, label=test_labels)
    test_params = {
        'tree_method': 'hist',
        'device': 'cuda',
        'verbosity': 0,
        'objective': 'binary:logistic'
    }
    xgb.train(test_params, dtrain_test, num_boost_round=1)
    print("XGBoost GPU Support: CONFIRME (hist + device=cuda fonctionne)")
except Exception as e:
    raise RuntimeError(f"ERREUR: XGBoost GPU non fonctionnel: {e}")

try:
    import cudf
    import cuml
    print(f"cuDF: {cudf.__version__}")
    print(f"cuML: {cuml.__version__}")
    print(f"CuPy: {cp.__version__}")
    print(f"GPU Memory Free: {cp.get_default_memory_pool().free_bytes()/1e9:.2f} GB")
except ImportError as e:
    raise RuntimeError(f"ERREUR: RAPIDS non installe: {e}")

print("\n" + "="*60)
print("KAGGLE GPU ENVIRONNEMENT PRET")
print("="*60)


VERIFICATION GPU OBLIGATOIRE - KAGGLE GPU CHALLENGE
name, memory.total [MiB], memory.free [MiB], driver_version, compute_cap
Tesla T4, 15360 MiB, 14912 MiB, 580.178.04, 7.5
Tesla T4, 15360 MiB, 14912 MiB, 580.178.04, 7.5

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0

PyTorch CUDA: 13.0
GPU Device: Tesla T4
GPU Memory: 15.6 GB
Compute Capability: (7, 5)


I0000 00:00:1795524206.109294      50 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TF GPU Devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
XGBoost Version: 3.4.1
XGBoost GPU Support: CONFIRME (hist + device=cuda fonctionne)
cuDF: 26.06.00
cuML: 26.06.00
CuPy: 14.0.1
GPU Memory Free: 0.00 GB

KAGGLE GPU ENVIRONNEMENT PRET


In [2]:
# ============================================================
# CELLULE 2: VERIFICATION/INSTALLATION RAPIDS KAGGLE (CUDA 13.0)
# ============================================================

!pip install -q --upgrade pip 2>/dev/null

# Core ML GPU packages
!pip install -q \
    xgboost==2.1.1 \
    lightgbm==4.3.0 \
    catboost==1.2.3 \
    shap==0.44.0 \
    optuna==3.6.0 \
    feature-engine==1.6.0 \
    category-encoders==2.6.0 \
    scikit-learn==1.4.0 \
    scipy==1.12.0 \
    numpy==1.26.0 \
    pandas==2.2.0 \
    polars==0.20.0 \
    numba==0.59.0 2>/dev/null

# Visualization & Explainability
!pip install -q plotly==5.20.0 matplotlib==3.8.0 seaborn==0.13.0 2>/dev/null
!pip install -q shap==0.44.0 lime==0.2.0 2>/dev/null

# ============================================================
# RAPIDS KAGGLE - VERIFICATION PRIORITAIRE (souvent pré-installé)
# ============================================================
import subprocess
import sys

print("="*60)
print("VERIFICATION RAPIDS KAGGLE (CUDA 13.0)")
print("="*60)

# 1. Check si RAPIDS déjà dispo (images Kaggle GPU récentes)
try:
    import cudf
    import cuml
    import cupy as cp
    print(f"cuDF: {cudf.__version__}")
    print(f"cuML: {cuml.__version__}")
    print(f"CuPy: {cp.__version__}")
    print(f"GPU Memory Free: {cp.get_default_memory_pool().free_bytes()/1e9:.2f} GB")
    print("RAPIDS DEJA INSTALLE - OK")
    HAS_RAPIDS = True
except ImportError:
    HAS_RAPIDS = False
    print("RAPIDS non trouvé")

# 2. Si pas dispo, essayer installation (mais CUDA 13.0 pas supporté officiellement)
if not HAS_RAPIDS:
    print("Tentative installation RAPIDS (CUDA 12.x packages sur CUDA 13.0)...")
    
    # Forcer CUDA 12.x car pas de packages cu13 encore
    pip_packages = [
        'cudf-cu12==24.06.*',
        'cuml-cu12==24.06.*',
        'cugraph-cu12==24.06.*',
        'rmm-cu12==24.06.*',
        'cupy-cuda12x==13.*'
    ]
    
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--extra-index-url=https://pypi.nvidia.com'] + pip_packages
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    
    if result.returncode == 0:
        try:
            import cudf
            import cuml
            import cupy as cp
            print(f"cuDF: {cudf.__version__}")
            print(f"cuML: {cuml.__version__}")
            print(f"CuPy: {cp.__version__}")
            print("RAPIDS INSTALLE (CUDA 12.x sur 13.0 - peut marcher)")
            HAS_RAPIDS = True
        except ImportError:
            HAS_RAPIDS = False
    else:
        print(f"Installation échouée: {result.stderr[-1000:]}")
        HAS_RAPIDS = False

# 3. Si toujours pas de RAPIDS, on continue quand même (XGBoost GPU + cuPy marchent)
if not HAS_RAPIDS:
    print("ATTENTION: RAPIDS non disponible - utilisation XGBoost GPU + cuPy + pandas/sklearn fallback")
    import cupy as cp
    print(f"CuPy: {cp.__version__}")
    print(f"GPU Memory Free: {cp.get_default_memory_pool().free_bytes()/1e9:.2f} GB")

print("="*60)
print("ENVIRONNEMENT KAGGLE PRET (RAPIDS optionnel)")
print("="*60)

globals()['HAS_RAPIDS'] = HAS_RAPIDS


  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... error
VERIFICATION RAPIDS KAGGLE (CUDA 13.0)
cuDF: 26.06.00
cuML: 26.06.00
CuPy: 14.0.1
GPU Memory Free: 0.00 GB
RAPIDS DEJA INSTALLE - OK
ENVIRONNEMENT KAGGLE PRET (RAPIDS optionnel)


In [52]:
# ============================================================
# CELLULE 3: CHARGEMENT DONNEES GOOGLE DRIVE AVEC cuDF (GPU)
# ============================================================

import cudf
import cupy as cp
import numpy as np
import pandas as pd
import time
import gc

print("="*60)
print("CHARGEMENT DATASET GOOGLE DRIVE SUR GPU (cuDF)")
print("="*60)

# Installer gdown si pas dispo
try:
    import gdown
except ImportError:
    !pip install -q gdown
    import gdown

# Télécharger depuis Google Drive
file_id = "17fIIactArdqnw9f7NSWgbQPyQYaioQo3"
output_path = "dataset_scoring_benin_2M.csv"

print(f"Téléchargement depuis Google Drive (file_id: {file_id})...")
gdown.download(f"https://drive.google.com/uc?id={file_id}", output_path, quiet=False)
print("Téléchargement terminé")

# Chargement cuDF (zero-copy GPU)
start = time.time()
df = cudf.read_csv(output_path)
load_time = time.time() - start

print(f"Temps de chargement cuDF: {load_time:.2f}s")
print(f"Shape: {df.shape}")
print(f"GPU Memory Used: {cp.get_default_memory_pool().used_bytes()/1e9:.2f} GB")

# Infos target
target_dist = df['defaut_paiement'].value_counts().to_pandas()
print(f"Distribution cible:")
print(target_dist)
print(f"Taux defaut: {target_dist[1]/target_dist.sum()*100:.1f}%")

# Colonnes categorielles vs numeriques
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
num_cols = [c for c in num_cols if c not in ['id_commercant', 'defaut_paiement']]

print(f"Colonnes categorielles ({len(cat_cols)}): {cat_cols}")
print(f"Colonnes numeriques ({len(num_cols)}): {num_cols}")

gc.collect()

globals()['df_raw'] = df
globals()['cat_cols'] = cat_cols
globals()['num_cols'] = num_cols


CHARGEMENT DATASET GOOGLE DRIVE SUR GPU (cuDF)
Téléchargement depuis Google Drive (file_id: 17fIIactArdqnw9f7NSWgbQPyQYaioQo3)...


Downloading...
From (original): https://drive.google.com/uc?id=17fIIactArdqnw9f7NSWgbQPyQYaioQo3
From (redirected): https://drive.google.com/uc?id=17fIIactArdqnw9f7NSWgbQPyQYaioQo3&confirm=t&uuid=116fe0fe-06ec-4a0e-b299-1ee3b9a92394
To: /kaggle/working/dataset_scoring_benin_2M.csv
100%|██████████| 230M/230M [00:01<00:00, 121MB/s]  


Téléchargement terminé
Temps de chargement cuDF: 0.20s
Shape: (700000, 42)
GPU Memory Used: 0.00 GB
Distribution cible:
defaut_paiement
0    455091
1    244909
Name: count, dtype: int64
Taux defaut: 35.0%
Colonnes categorielles (6): ['id_commercant', 'genre', 'niveau_education', 'situation_familiale', 'departement', 'secteur_activite']
Colonnes numeriques (35): ['age', 'anciennete_activite_mois', 'revenu_mensuel_declare', 'depenses_mensuelles', 'proprietaire_local', 'nombre_employes', 'nb_transactions_mois', 'volume_depots_moyen', 'volume_retraits_moyen', 'solde_moyen_compte', 'tendance_solde_3m', 'regularite_depots_ecart_type', 'nb_contreparties_distinctes', 'part_transferts_marchands', 'nb_transactions_echouees', 'frequence_recharges_mois', 'montant_moyen_recharge', 'anciennete_ligne_mois', 'nb_contacts_distincts', 'stabilite_lieu_vie', 'membre_tontine', 'anciennete_tontine_mois', 'regularite_cotisations_tontine', 'montant_moyen_cotisation', 'deja_beneficiaire_tour', 'score_aptitude_

In [53]:
# ============================================================
# CELLULE 4: FEATURE ENGINEERING MASSIF SUR GPU (cuDF/cuPy)
# ============================================================

import cudf
import cupy as cp
import numpy as np
import time
import gc

df = globals()['df_raw'].copy()

print("="*60)
print("FEATURE ENGINEERING INTENSIF SUR GPU")
print("="*60)

start = time.time()

# ============================================================
# 1. RATIOS FINANCIERS CRITIQUES
# ============================================================
print("1. Ratios financiers...")
df['ratio_endettement'] = df['montant_demande'] / (df['revenu_mensuel_declare'] + 1)
df['charge_mensuelle'] = df['montant_demande'] / (df['duree_credit_mois'] + 1)
df['ratio_charge_revenu'] = df['charge_mensuelle'] / (df['revenu_mensuel_declare'] + 1)
df['ratio_depenses_revenus'] = df['depenses_mensuelles'] / (df['revenu_mensuel_declare'] + 1)
df['marge_mensuelle'] = df['revenu_mensuel_declare'] - df['depenses_mensuelles'] - df['charge_mensuelle']
df['ratio_marge_revenu'] = df['marge_mensuelle'] / (df['revenu_mensuel_declare'] + 1)
df['capacite_remboursement'] = df['revenu_mensuel_declare'] / (df['charge_mensuelle'] + 1)
df['dette_revenu_annuel'] = df['montant_demande'] / (df['revenu_mensuel_declare'] * 12 + 1)

# ============================================================
# 2. FEATURES MOBILE MONEY AVANCEES
# ============================================================
print("2. Features Mobile Money...")
df['ratio_depots_retraits'] = df['volume_depots_moyen'] / (df['volume_retraits_moyen'] + 1)
df['taux_echec_transactions'] = df['nb_transactions_echouees'] / (df['nb_transactions_mois'] + 1)
df['solde_par_transaction'] = df['solde_moyen_compte'] / (df['nb_transactions_mois'] + 1)
df['volume_total_mensuel'] = df['volume_depots_moyen'] + df['volume_retraits_moyen']
df['net_flow_mensuel'] = df['volume_depots_moyen'] - df['volume_retraits_moyen']
df['ratio_transferts_marchands'] = df['part_transferts_marchands']
df['diversite_contreparties'] = df['nb_contreparties_distinctes'] / (df['nb_transactions_mois'] + 1)
df['regularite_score'] = 1 / (1 + df['regularite_depots_ecart_type'])
df['croissance_solde_3m'] = df['tendance_solde_3m'] / (df['solde_moyen_compte'].abs() + 1)
df['intensite_activite'] = df['nb_transactions_mois'] * df['volume_depots_moyen']
df['concentration_risque'] = 1 - df['part_transferts_marchands']
df['liquidite_immediate'] = df['solde_moyen_compte'] / (df['charge_mensuelle'] + 1)

# ============================================================
# 3. FEATURES TELECOM & MOBILITE
# ============================================================
print("3. Features Telecom...")
df['recharge_par_contact'] = df['frequence_recharges_mois'] / (df['nb_contacts_distincts'] + 1)
df['budget_telecom_mensuel'] = df['frequence_recharges_mois'] * df['montant_moyen_recharge']
df['ratio_telecom_revenu'] = df['budget_telecom_mensuel'] / (df['revenu_mensuel_declare'] + 1)
df['stabilite_composite'] = df['stabilite_lieu_vie'] * (df['anciennete_ligne_mois'] / 240.0)
df['anciennete_ligne_annees'] = df['anciennete_ligne_mois'] / 12.0
df['activite_telecom_score'] = cp.log1p(df['frequence_recharges_mois']) * cp.log1p(df['nb_contacts_distincts'])

# ============================================================
# 4. FEATURES TONTINE & EPARGNE INFORMELLE
# ============================================================
print("4. Features Tontine...")
df['tontine_score_composite'] = df['membre_tontine'] * df['regularite_cotisations_tontine'] * cp.log1p(df['anciennete_tontine_mois'])
df['capacite_epargne_mensuelle'] = df['membre_tontine'] * df['montant_moyen_cotisation']
df['ratio_epargne_revenu'] = df['capacite_epargne_mensuelle'] / (df['revenu_mensuel_declare'] + 1)
df['experience_gestion_somme'] = df['deja_beneficiaire_tour'].astype('float32')
df['discipline_financiere'] = df['regularite_cotisations_tontine'] * df['membre_tontine']

# ============================================================
# 5. FEATURES PSYCHOMETRIQUES
# ============================================================
print("5. Features Psychometriques...")
df['score_entrepreneur_normalise'] = df['score_aptitude_entrepreneuriale'] / 100.0
df['score_integrite_normalise'] = df['score_integrite'] / 100.0
df['locus_controle_normalise'] = df['locus_controle_interne'] / 10.0
df['profil_psychologique_composite'] = (
    df['score_entrepreneur_normalise'] * 0.4 +
    df['score_integrite_normalise'] * 0.4 +
    df['locus_controle_normalise'] * 0.2
)
df['risque_comportemental'] = 1 - df['profil_psychologique_composite']

# ============================================================
# 6. FEATURES CONTEXTUELLES
# ============================================================
print("6. Features Contextuelles...")
df['proximite_marche_score'] = 1 / (1 + df['proximite_marche_km'])
df['jour_marche_bonus'] = df['indicateur_jour_marche'].astype('float32')
df['groupement_pro_bonus'] = df['membre_groupement_pro'].astype('float32')
df['stabilite_geographique'] = df['stabilite_lieu_vie']

# ============================================================
# 7. FEATURES DEMOGRAPHIQUES
# ============================================================
print("7. Features Demographiques...")
df['age_groupe'] = cudf.cut(df['age'], bins=[18, 25, 35, 45, 55, 70], labels=[1,2,3,4,5]).astype('float32')
df['anciennete_activite_annees'] = df['anciennete_activite_mois'] / 12.0
df['maturite_entreprise'] = cp.log1p(df['anciennete_activite_mois'])
df['taille_entreprise'] = cp.log1p(df['nombre_employes'])
df['proprietaire_bonus'] = df['proprietaire_local'].astype('float32')
df['revenu_par_employe'] = df['revenu_mensuel_declare'] / (df['nombre_employes'] + 1)

# ============================================================
# 8. FEATURES DE RISQUE
# ============================================================
print("8. Features Risque...")
df['primo_emprunteur'] = df['premiere_fois'].astype('float32')
df['garantie_bonus'] = df['garantie'].astype('float32')
df['risque_primo_sans_garantie'] = df['primo_emprunteur'] * (1 - df['garantie_bonus'])
df['duree_credit_annees'] = df['duree_credit_mois'] / 12.0
df['montant_demande_log'] = cp.log1p(df['montant_demande'])
df['revenu_log'] = cp.log1p(df['revenu_mensuel_declare'])

# ============================================================
# 9. INTERACTIONS CROISEES
# ============================================================
print("9. Interactions croisees...")
df['interaction_revenu_anciennete'] = df['revenu_mensuel_declare'] * cp.log1p(df['anciennete_activite_mois'])
df['interaction_solde_tendance'] = df['solde_moyen_compte'] * df['tendance_solde_3m']
df['interaction_tontine_integrite'] = df['tontine_score_composite'] * df['score_integrite_normalise']
df['interaction_psycho_financier'] = df['profil_psychologique_composite'] * df['capacite_remboursement']
df['interaction_activite_stabilite'] = df['intensite_activite'] * df['stabilite_composite']
df['interaction_garantie_revenu'] = df['garantie_bonus'] * df['revenu_mensuel_declare']
df['interaction_age_experience'] = df['age'] * df['anciennete_activite_annees']
df['interaction_charge_liquidite'] = df['charge_mensuelle'] / (df['solde_moyen_compte'] + 1)
df['interaction_diversite_regularite'] = df['diversite_contreparties'] * df['regularite_score']
df['interaction_telecom_revenu'] = df['budget_telecom_mensuel'] * df['ratio_depots_retraits']

# ============================================================
# 10. FEATURES POLYNOMIALES
# ============================================================
print("10. Features polynomiales...")
for col in ['ratio_endettement', 'charge_mensuelle', 'solde_moyen_compte', 'revenu_mensuel_declare', 'montant_demande']:
    df[f'{col}_sq'] = df[col] ** 2
    df[f'{col}_sqrt'] = cp.sqrt(cp.maximum(df[col].values, 0))
    df[f'{col}_log'] = cp.log1p(cp.maximum(df[col].values, 0))

# ============================================================
# 11. TARGET ENCODING CATEGORIEL
# ============================================================
print("11. Target Encoding categoriel...")
cat_features = ['genre', 'niveau_education', 'situation_familiale', 'departement', 'secteur_activite']
target = df['defaut_paiement']
global_mean = target.mean()

for col in cat_features:
    stats = df.groupby(col)['defaut_paiement'].agg(['mean', 'count'])
    smoothing = 100
    stats['te'] = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
    te_map = stats['te'].to_dict()
    df[f'{col}_te'] = df[col].map(te_map).fillna(global_mean)
    freq_map = stats['count'].to_dict()
    df[f'{col}_freq'] = df[col].map(freq_map).fillna(0)

# One-hot pour variables a faible cardinalite
for col in ['genre', 'proprietaire_local', 'membre_tontine', 'indicateur_jour_marche', 'membre_groupement_pro', 'premiere_fois', 'garantie']:
    if col in df.columns:
        dummies = cudf.get_dummies(df[col], prefix=col, dtype='float32')
        df = cudf.concat([df, dummies], axis=1)

feat_time = time.time() - start
print(f"Feature Engineering GPU: {feat_time:.2f}s")
print(f"Nouveau shape: {df.shape}")
print(f"GPU Memory: {cp.get_default_memory_pool().used_bytes()/1e9:.2f} GB")

exclude_cols = ['id_commercant', 'defaut_paiement'] + cat_features
feature_cols = [c for c in df.columns if c not in exclude_cols]
print(f"Features finales: {len(feature_cols)}")

gc.collect()

globals()['df_fe'] = df
globals()['feature_cols_final'] = feature_cols 


# Après chaque gros bloc, libérer la mémoire
cp.get_default_memory_pool().free_all_blocks()
gc.collect()


FEATURE ENGINEERING INTENSIF SUR GPU
1. Ratios financiers...
2. Features Mobile Money...
3. Features Telecom...
4. Features Tontine...
5. Features Psychometriques...
6. Features Contextuelles...
7. Features Demographiques...
8. Features Risque...
9. Interactions croisees...
10. Features polynomiales...
11. Target Encoding categoriel...
Feature Engineering GPU: 0.58s
Nouveau shape: (700000, 142)
GPU Memory: 0.00 GB
Features finales: 135


0

In [54]:
# ============================================================
# CELLULE 5: PREPROCESSING & SPLIT STRATIFIE SUR GPU (cuML)
# ============================================================

import cudf
import cupy as cp
import numpy as np
from cuml.model_selection import train_test_split
from cuml.preprocessing import RobustScaler
import gc
import xgboost as xgb

df_fe = globals()['df_fe']
feature_cols_final = globals()['feature_cols_final']

print("="*60)
print("PREPROCESSING & SPLIT GPU (cuML)")
print("="*60)

X = df_fe[feature_cols_final].astype('float32')
y = df_fe['defaut_paiement'].astype('float32')

X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

# Split stratifie sur GPU
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_pandas().to_dict()}")
print(f"Test target dist: {y_test.value_counts().to_pandas().to_dict()}")

# RobustScaler sur GPU
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# DMatrix pour XGBoost GPU (zero-copy)
dtrain = xgb.DMatrix(X_train_scaled, label=y_train, feature_names=feature_cols_final)
dtest = xgb.DMatrix(X_test_scaled, label=y_test, feature_names=feature_cols_final)

print(f"DMatrix creees sur GPU")
print(f"GPU Memory: {cp.get_default_memory_pool().used_bytes()/1e9:.2f} GB")

gc.collect()

globals()['X_train'] = X_train
globals()['X_test'] = X_test
globals()['y_train'] = y_train
globals()['y_test'] = y_test
globals()['X_train_scaled'] = X_train_scaled
globals()['X_test_scaled'] = X_test_scaled
globals()['dtrain'] = dtrain
globals()['dtest'] = dtest
globals()['scaler'] = scaler

PREPROCESSING & SPLIT GPU (cuML)
Train: (560000, 135), Test: (140000, 135)
Train target dist: {0.0: 364073, 1.0: 195927}
Test target dist: {0.0: 91018, 1.0: 48982}
DMatrix creees sur GPU
GPU Memory: 0.00 GB


In [55]:
# ============================================================
# CELLULE 6: XGBOOST ENTRAINEMENT GPU (tree_method='hist' + device='cuda')
# ============================================================

import xgboost as xgb
import cupy as cp
import numpy as np
import pandas as pd
import time
import gc
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

dtrain = globals()['dtrain']
dtest = globals()['dtest']
y_test = globals()['y_test']

print("="*60)
print("XGBOOST ENTRAINEMENT GPU - hist + device='cuda' (XGBoost 3.x)")
print("="*60)

# Calculer scale_pos_weight a partir des donnees d'entrainement
y_train = globals()['y_train']
pos_count = (y_train == 1).sum()
neg_count = (y_train == 0).sum()
scale_pos_weight = float(neg_count / pos_count) if pos_count > 0 else 1.0
print(f"Class balance: pos={pos_count}, neg={neg_count}, scale_pos_weight={scale_pos_weight:.4f}")

params = {
    'tree_method': 'hist',               # Histogramme (GPU via device='cuda')
    'device': 'cuda',                    # GPU device (remplace gpu_id/gpu_predictor)
    'max_bin': 512,
    'single_precision_histogram': True,
    
    'objective': 'binary:logistic',
    'eval_metric': ['auc', 'logloss', 'error'],
    
    'lambda': 2.0,
    'alpha': 0.5,
    'gamma': 0.1,
    
    'max_depth': 10,
    'min_child_weight': 50,
    'max_delta_step': 1,
    
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'colsample_bylevel': 0.8,
    'colsample_bynode': 0.8,
    
    'eta': 0.03,
    'scale_pos_weight': scale_pos_weight,
    'verbosity': 1,
    
    'grow_policy': 'lossguide',
    'max_leaves': 255,
}

print("Configuration GPU:")
for k, v in params.items():
    print(f"  {k}: {v}")

evals = [(dtrain, 'train'), (dtest, 'eval')]
evals_result = {}

start = time.time()

model = xgb.train(
    params,
    dtrain,
    num_boost_round=3000,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=100,
    verbose_eval=50,
    maximize=True
)

train_time = time.time() - start
print(f"\nTemps entrainement GPU: {train_time:.2f}s")
print(f"Meilleur iteration: {model.best_iteration}")
print(f"Meilleur AUC: {model.best_score:.6f}")

# Monitoring GPU pendant entrainement
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True)
if result.stdout:
    parts = result.stdout.strip().split(', ')
    if len(parts) >= 3:
        gpu_util, mem_used, mem_total = parts[0], parts[1], parts[2]
        print(f"GPU Utilization: {gpu_util}% | VRAM: {mem_used}/{mem_total} MB")
    else:
        print(f"nvidia-smi output: {result.stdout.strip()}")

y_pred_proba = model.predict(dtest)
y_pred = (y_pred_proba > 0.5).astype(int)

y_test_np = y_test.to_numpy()
auc = roc_auc_score(y_test_np, y_pred_proba)
acc = accuracy_score(y_test_np, y_pred)
prec = precision_score(y_test_np, y_pred)
rec = recall_score(y_test_np, y_pred)
f1 = f1_score(y_test_np, y_pred)

print(f"\nMETRIQUES TEST:")
print(f"  AUC-ROC: {auc:.6f}")
print(f"  Accuracy: {acc:.6f}")
print(f"  Precision: {prec:.6f}")
print(f"  Recall: {rec:.6f}")
print(f"  F1-Score: {f1:.6f}")

importance = model.get_score(importance_type='gain')
importance_df = pd.DataFrame(list(importance.items()), columns=['feature', 'gain'])
importance_df = importance_df.sort_values('gain', ascending=False)
print(f"\nTOP 20 FEATURES:")
print(importance_df.head(20).to_string())

gc.collect()
globals()['xgb_model'] = model
globals()['importance_df'] = importance_df
globals()['evals_result'] = evals_result


XGBOOST ENTRAINEMENT GPU - hist + device='cuda' (XGBoost 3.x)
Class balance: pos=195927, neg=364073, scale_pos_weight=1.8582
Configuration GPU:
  tree_method: hist
  device: cuda
  max_bin: 512
  single_precision_histogram: True
  objective: binary:logistic
  eval_metric: ['auc', 'logloss', 'error']
  lambda: 2.0
  alpha: 0.5
  gamma: 0.1
  max_depth: 10
  min_child_weight: 50
  max_delta_step: 1
  subsample: 0.8
  colsample_bytree: 0.8
  colsample_bylevel: 0.8
  colsample_bynode: 0.8
  eta: 0.03
  scale_pos_weight: 1.858207393570054
  verbosity: 1
  grow_policy: lossguide
  max_leaves: 255


/usr/local/lib/python3.13/dist-packages/xgboost/callback.py:385: UserWarning:

[14:47:34] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "single_precision_histogram" } are not used.




[0]	train-auc:0.99947	train-logloss:0.67829	train-error:0.00098	eval-auc:0.99943	eval-logloss:0.67829	eval-error:0.00106
[50]	train-auc:1.00000	train-logloss:0.19707	train-error:0.00037	eval-auc:1.00000	eval-logloss:0.19714	eval-error:0.00039
[100]	train-auc:1.00000	train-logloss:0.04823	train-error:0.00024	eval-auc:1.00000	eval-logloss:0.04826	eval-error:0.00021
[101]	train-auc:1.00000	train-logloss:0.04686	train-error:0.00024	eval-auc:1.00000	eval-logloss:0.04689	eval-error:0.00021

Temps entrainement GPU: 1.76s
Meilleur iteration: 1
Meilleur AUC: 0.001279
GPU Utilization: 82% | VRAM: 4477/15360
0 MB

METRIQUES TEST:
  AUC-ROC: 1.000000
  Accuracy: 0.999786
  Precision: 0.999449
  Recall: 0.999939
  F1-Score: 0.999694

TOP 20 FEATURES:
                           feature           gain
36         solde_moyen_compte_sqrt  160526.703125
4               solde_moyen_compte  147454.843750
37          solde_moyen_compte_log   69593.914062
35           solde_moyen_compte_sq   63326.011719
24

In [56]:
# ============================================================
# CELLULE 7: SHAP GPU - EXPLICABILITE ACCELEREE
# ============================================================

import shap
import cupy as cp
import numpy as np
import pandas as pd
import time
import gc
import matplotlib.pyplot as plt

xgb_model = globals()['xgb_model']
X_test_scaled = globals()['X_test_scaled']
y_test = globals()['y_test']
feature_cols_final = globals()['feature_cols_final']

print("="*60)
print("SHAP GPU - EXPLICABILITE ACCELEREE")
print("="*60)

sample_size = 10000
X_test_sample = X_test_scaled[:sample_size]
y_test_sample = y_test[:sample_size]

print(f"Echantillon SHAP: {sample_size} observations")

start = time.time()
explainer = shap.TreeExplainer(
    xgb_model,
    data=X_test_sample[:500].to_pandas(),  # background dataset (pandas)
    model_output="probability",
    feature_perturbation="interventional",
)
print(f"Explainer init: {time.time() - start:.2f}s")

start = time.time()
shap_values = explainer.shap_values(X_test_sample)
shap_time = time.time() - start
print(f"SHAP values GPU: {shap_time:.2f}s")
print(f"GPU Memory: {cp.get_default_memory_pool().used_bytes()/1e9:.2f} GB")

# Visualisations
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_test_sample.to_pandas(), feature_names=feature_cols_final, show=False, max_display=30)
plt.title(f'SHAP Summary - Top 30 Features (n={sample_size})', fontsize=14)
plt.tight_layout()
plt.show()

mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'feature': feature_cols_final,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

plt.figure(figsize=(12, 8))
top20 = shap_importance.head(20)
plt.barh(range(len(top20)), top20['mean_abs_shap'][::-1])
plt.yticks(range(len(top20)), top20['feature'][::-1])
plt.xlabel('Mean |SHAP Value|')
plt.title('Top 20 Features by SHAP Importance', fontsize=14)
plt.tight_layout()
plt.show()

top5_features = shap_importance.head(5)['feature'].tolist()
for feat in top5_features:
    plt.figure(figsize=(8, 6))
    shap.dependence_plot(feat, shap_values, X_test_sample.to_pandas(), feature_names=feature_cols_final, show=False)
    plt.title(f'SHAP Dependence: {feat}')
    plt.tight_layout()
    plt.show()

idx = 0
shap.force_plot(explainer.expected_value, shap_values[idx], X_test_sample.iloc[idx].to_pandas(), feature_names=feature_cols_final, matplotlib=True, show=False)
plt.title(f'SHAP Force Plot - Sample {idx}')
plt.tight_layout()
plt.show()

print("SHAP ANALYSIS COMPLETE")
gc.collect()

globals()['shap_values'] = shap_values
globals()['shap_importance'] = shap_importance
globals()['explainer'] = explainer


Background dataset has 500 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=500 when initializing the masker.


SHAP GPU - EXPLICABILITE ACCELEREE
Echantillon SHAP: 10000 observations
Explainer init: 0.04s


AttributeError: DataFrame object has no attribute dtype

In [58]:
# ============================================================
# CELLULE 8: REGLES METIER IMF BENIN - SCORING 0-100 & RECOMMANDATIONS
# ============================================================

import numpy as np
import pandas as pd

print("="*60)
print("REGLES METIER IMF BENIN - MOTEUR DE SCORING")
print("="*60)

# ============================================================
# CONSTANTES METIER (Taux usure 27% au Benin)
# ============================================================
TAUX_USURE = 0.27
PLAFOND_PRIMO_MIN = 10000
PLAFOND_PRIMO_MAX = 50000
PLAFOND_RENOUVELLEMENT_FACTEUR = 2.0
DUREE_MAX_MOIS = 36
CHARGE_MAX_RATIO = 0.33

def compute_business_score(row, proba_default):
    """
    Convertit probabilite de defaut en score 0-100
    Applique les regles metier IMF
    """
    base_score = (1 - proba_default) * 100
    adjustments = 0
    
    if row['premiere_fois'] == 1:
        adjustments -= 10
    if row['garantie'] == 1:
        adjustments += 15
    if row['membre_tontine'] == 1 and row['regularite_cotisations_tontine'] > 0.8:
        adjustments += 10
    if row['ratio_endettement'] > 0.5:
        adjustments -= 20
    elif row['ratio_endettement'] > 0.3:
        adjustments -= 10
    if row['ratio_charge_revenu'] > 0.33:
        adjustments -= 15
    if row['solde_moyen_compte'] < row['charge_mensuelle'] * 3:
        adjustments -= 10
    if row['nb_transactions_echouees'] > 2:
        adjustments -= 5 * min(row['nb_transactions_echouees'], 10)
    if row['stabilite_lieu_vie'] > 0.8 and row['anciennete_ligne_mois'] > 24:
        adjustments += 5
    if row['profil_psychologique_composite'] > 0.7:
        adjustments += 8
    if row['anciennete_activite_mois'] > 24:
        adjustments += 5
    
    final_score = np.clip(base_score + adjustments, 0, 100)
    return final_score

def recommend_credit_product(row, score, proba_default):
    """Recommande produit, montant, duree selon regles IMF"""
    revenu = row['revenu_mensuel_declare']
    charge_max = revenu * CHARGE_MAX_RATIO
    
    is_primo = row['premiere_fois'] == 1
    has_garantie = row['garantie'] == 1
    
    if is_primo:
        plafond = PLAFOND_PRIMO_MAX if has_garantie else PLAFOND_PRIMO_MIN
        plafond = int(plafond * (score / 100))
    else:
        base_plafond = min(row['montant_demande'] * PLAFOND_RENOUVELLEMENT_FACTEUR, revenu * 12)
        plafond = int(base_plafond * (score / 100))
    
    duree_optimale = min(int(plafond / charge_max), DUREE_MAX_MOIS)
    duree_optimale = max(duree_optimale, 3)
    
    mensualite = plafond / duree_optimale if duree_optimale > 0 else 0
    taux_effectif = min(0.24, TAUX_USURE * 0.9)
    
    if is_primo and score < 50:
        produit = "Micro-credit Primo (Garantie Solidaire)"
    elif is_primo:
        produit = "Credit Primo Standard"
    elif score >= 75:
        produit = "Credit Renouvellement Premium"
    elif score >= 60:
        produit = "Credit Renouvellement Standard"
    else:
        produit = "Credit de Groupe / Tontine"
    
    if score >= 70:
        decision = "APPROUVE"
    elif score >= 50:
        decision = "CONDITIONNEL (Garantie requise)"
    else:
        decision = "REFUSE"
    
    return {
        'score': round(score, 1),
        'decision': decision,
        'produit': produit,
        'montant_recommande': int(plafond),
        'duree_mois': duree_optimale,
        'mensualite_estimee': round(mensualite, 0),
        'taux_effectif': round(taux_effectif * 100, 1),
        'charge_ratio': round(mensualite / revenu * 100, 1) if revenu > 0 else 0
    }

def generate_explanation(row, shap_vals, feature_names, top_n=5):
    """Genere explication textuelle depuis SHAP"""
    if shap_vals is None:
        return "Explication non disponible"
    abs_shap = np.abs(shap_vals)
    top_idx = np.argsort(abs_shap)[-top_n:][::-1]
    factors = []
    for idx in top_idx:
        feat = feature_names[idx]
        val = shap_vals[idx]
        direction = "favorable" if val > 0 else "defavorable"
        factors.append(f"{feat} ({direction})")
    return f"Score {'eleve' if row['score'] > 60 else 'modere'} car " + ", ".join(factors[:3]) + "."

REGLES METIER IMF BENIN - MOTEUR DE SCORING


In [22]:
# ============================================================
# CELLULE 9: APPLICATION MOTEUR DE SCORING SUR TEST SET (GPU)
# ============================================================

import numpy as np
import pandas as pd
import time

xgb_model = globals()['xgb_model']
dtest = globals()['dtest']
df_fe = globals()['df_fe']
X_test = globals()['X_test']
y_test = globals()['y_test']
compute_business_score = globals()['compute_business_score']
recommend_credit_product = globals()['recommend_credit_product']

print("="*60)
print("APPLICATION SCORING 0-100 + RECOMMANDATIONS - GPU")
print("="*60)

start = time.time()

y_pred_proba = xgb_model.predict(dtest)

test_df = df_fe.iloc[X_test.index.to_pandas()].to_pandas().reset_index(drop=True)
test_df['proba_default'] = y_pred_proba

n = len(test_df)
scores = np.zeros(n)
recommendations = []

for i in range(n):
    row = test_df.iloc[i]
    score = compute_business_score(row, row['proba_default'])
    rec = recommend_credit_product(row, score, row['proba_default'])
    scores[i] = score
    recommendations.append(rec)
    
    if i % 50000 == 0 and i > 0:
        print(f"  Processed: {i}/{n}")

test_df['score_0_100'] = scores
rec_df = pd.DataFrame(recommendations)
test_df = pd.concat([test_df.reset_index(drop=True), rec_df], axis=1)

print(f"Scoring complet: {time.time() - start:.2f}s")

print(f"DISTRIBUTION SCORES 0-100:")
print(test_df['score_0_100'].describe())

print(f"DECISIONS:")
print(test_df['decision'].value_counts())

print(f"PRODUITS RECOMMANDES:")
print(test_df['produit'].value_counts())

print(f"MONTANT MOYEN RECOMMANDE: {test_df['montant_recommande'].mean():,.0f} FCFA")
print(f"DUREE MOYENNE: {test_df['duree_mois'].mean():.1f} mois")

corr = test_df['score_0_100'].corr(test_df['defaut_paiement'])
print(f"Correlation Score vs Defaut: {corr:.4f}")

print(f"EXEMPLES INDIVIDUELS:")
for idx in [0, 100, 1000, 5000]:
    if idx < len(test_df):
        row = test_df.iloc[idx]
        print(f"  ID: {row['id_commercant']} | Score: {row['score_0_100']:.1f} | {row['decision']} | {row['produit']} | {row['montant_recommande']:,.0f} FCFA / {row['duree_mois']}m")

import gc
gc.collect()
globals()['scored_df'] = test_df


APPLICATION SCORING 0-100 + RECOMMANDATIONS - GPU
  Processed: 50000/400000
  Processed: 100000/400000
  Processed: 150000/400000
  Processed: 200000/400000
  Processed: 250000/400000
  Processed: 300000/400000
  Processed: 350000/400000
Scoring complet: 79.18s
DISTRIBUTION SCORES 0-100:
count    400000.000000
mean         20.579459
std          16.243552
min           0.000000
25%           6.688471
50%          18.998026
75%          32.278336
max          90.194901
Name: score_0_100, dtype: float64
DECISIONS:
decision
REFUSE                             380291
CONDITIONNEL (Garantie requise)     18868
APPROUVE                              841
Name: count, dtype: int64
PRODUITS RECOMMANDES:
produit
Micro-credit Primo (Garantie Solidaire)    237640
Credit de Groupe / Tontine                 155147
Credit Renouvellement Standard               4581
Credit Primo Standard                        2359
Credit Renouvellement Premium                 273
Name: count, dtype: int64
MONTANT MOYEN R

In [59]:
# ============================================================
# CELLULE 10: BENCHMARK GPU vs CPU - DEMONSTRATION ACCELERATION GPU
# ============================================================

import xgboost as xgb
import cupy as cp
import numpy as np
import pandas as pd
import time
import gc
import subprocess

X_train = globals()['X_train']
y_train = globals()['y_train']
X_test = globals()['X_test']
y_test = globals()['y_test']
dtrain = globals()['dtrain']
dtest = globals()['dtest']
X_test_scaled = globals()['X_test_scaled']
feature_cols_final = globals()['feature_cols_final']

print("="*60)
print("BENCHMARK GPU vs CPU - PREUVE D'ACCELERATION GPU")
print("="*60)

def to_numpy(x):
    if hasattr(x, 'to_pandas'):
        return x.to_pandas().values
    elif hasattr(x, 'to_numpy'):
        return x.to_numpy()
    return x

# Preparer donnees CPU
X_train_cpu = to_numpy(X_train).astype('float32')
y_train_cpu = to_numpy(y_train).astype('float32')
X_test_cpu = to_numpy(X_test).astype('float32')
y_test_cpu = to_numpy(y_test).astype('float32')

dtrain_cpu = xgb.DMatrix(X_train_cpu, label=y_train_cpu)
dtest_cpu = xgb.DMatrix(X_test_cpu, label=y_test_cpu)

base_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 8,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 1300000/700000,
    'lambda': 2.0,
    'alpha': 0.5,
    'verbosity': 0,
}

n_rounds = 500

# ============================================================
# CPU TRAINING (baseline)
# ============================================================
print("CPU Training (tree_method='hist')...")
cpu_params = base_params.copy()
cpu_params['tree_method'] = 'hist'
cpu_params['predictor'] = 'cpu_predictor'
cpu_params['nthread'] = -1

gc.collect()
start = time.time()
cpu_model = xgb.train(cpu_params, dtrain_cpu, num_boost_round=n_rounds, verbose_eval=False)
cpu_time = time.time() - start
cpu_auc = cpu_model.eval(dtest_cpu)
print(f"  CPU Time: {cpu_time:.2f}s | AUC: {float(cpu_auc.split(':')[1]):.4f}")

# ============================================================
# GPU TRAINING (notre config optimisee)
# ============================================================
print("GPU Training (tree_method='hist' + device='cuda')...")
gpu_params = base_params.copy()
gpu_params['tree_method'] = 'hist'
gpu_params['device'] = 'cuda'
# gpu_id removed in XGBoost 3.x
gpu_params['max_bin'] = 256
# single_precision_histogram not supported with device='cuda'

gc.collect()
cp.get_default_memory_pool().free_all_blocks()
start = time.time()
gpu_model = xgb.train(gpu_params, dtrain, num_boost_round=n_rounds, verbose_eval=False)
gpu_time = time.time() - start
gpu_auc = gpu_model.eval(dtest)
print(f"  GPU Time: {gpu_time:.2f}s | AUC: {float(gpu_auc.split(':')[1]):.4f}")

# ============================================================
# RESULTATS - SPEEDUP GPU
# ============================================================
speedup = cpu_time / gpu_time
print(f"\nSPEEDUP GPU: {speedup:.2f}x PLUS VITE")
print(f"GPU Memory Peak: {cp.get_default_memory_pool().used_bytes()/1e9:.2f} GB")

# nvidia-smi pendant benchmark
result = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu', '--format=csv,noheader,nounits'], capture_output=True, text=True)
if result.stdout:
    parts = result.stdout.strip().split(", ")
    if len(parts) >= 4:
        gpu_util, mem_used, mem_total, temp = parts[0], parts[1], parts[2], parts[3]
        print(f"GPU Utilization: {gpu_util}% | VRAM: {mem_used}/{mem_total} MB | Temp: {temp}C")
    else:
        print(f"nvidia-smi output: {result.stdout.strip()}")

# Benchmark prediction
n_pred = 100000
X_pred = X_test_scaled[:n_pred]
X_pred_cpu = to_numpy(X_pred)

start = time.time()
_ = cpu_model.predict(xgb.DMatrix(X_pred_cpu))
cpu_pred_time = time.time() - start

start = time.time()
_ = gpu_model.predict(xgb.DMatrix(X_pred, feature_names=feature_cols_final))
gpu_pred_time = time.time() - start

print(f"\nPREDICTION BENCHMARK ({n_pred:,} samples):")
print(f"  CPU: {cpu_pred_time*1000:.1f}ms")
print(f"  GPU: {gpu_pred_time*1000:.1f}ms")
print(f"  Speedup prediction: {cpu_pred_time/gpu_pred_time:.2f}x")

gc.collect()


BENCHMARK GPU vs CPU - PREUVE D'ACCELERATION GPU
CPU Training (tree_method='hist')...
  CPU Time: 35.42s | AUC: 1.0000
GPU Training (tree_method='hist' + device='cuda')...
  GPU Time: 4.54s | AUC: 1.0000

SPEEDUP GPU: 7.80x PLUS VITE
GPU Memory Peak: 0.00 GB
GPU Utilization: 87% | VRAM: 4399/15360 MB | Temp: 78
0C

PREDICTION BENCHMARK (100,000 samples):
  CPU: 413.7ms
  GPU: 70.2ms
  Speedup prediction: 5.89x


32

In [28]:
# ============================================================
# CELLULE 11: OPTUNA HYPERPARAMETER TUNING GPU (hist + device='cuda')
# ============================================================

import optuna
import xgboost as xgb
import time
import gc

dtrain = globals()['dtrain']

print("="*60)
print("OPTUNA HYPERPARAMETER TUNING GPU - hist + device='cuda'")
print("="*60)

def objective(trial):
    params = {
        'tree_method': 'hist',
        'device': 'cuda',
        # gpu_id removed in XGBoost 3.x
        # single_precision_histogram not supported with device='cuda'
        'objective': 'binary:logistic',
        'eval_metric': 'auc',
        'verbosity': 0,
        'scale_pos_weight': 1300000/700000,
        
        'max_depth': trial.suggest_int('max_depth', 6, 14),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'lambda': trial.suggest_float('lambda', 0.1, 10.0, log=True),
        'alpha': trial.suggest_float('alpha', 0.0, 5.0),
        'gamma': trial.suggest_float('gamma', 0.0, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 10, 200),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 5),
        'max_bin': trial.suggest_categorical('max_bin', [128, 256, 512, 1024]),
        'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']),
        'max_leaves': trial.suggest_int('max_leaves', 31, 511),
    }
    
    cv_result = xgb.cv(
        params,
        dtrain,
        num_boost_round=1000,
        nfold=3,
        early_stopping_rounds=50,
        maximize=True,
        verbose_eval=False,
        seed=42
    )
    
    best_auc = cv_result['test-auc-mean'].max()
    return best_auc

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=10),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=20)
)

print("Lancement optimisation GPU (50 trials, hist + device='cuda')...")
start = time.time()
study.optimize(objective, n_trials=50, timeout=3600, show_progress_bar=True)
opt_time = time.time() - start

print(f"\nTemps optimisation GPU: {opt_time/60:.1f} min")
print(f"Meilleur AUC: {study.best_value:.6f}")
print(f"Meilleurs params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

try:
    fig = optuna.visualization.plot_optimization_history(study)
    fig.show()
    fig = optuna.visualization.plot_param_importances(study)
    fig.show()
except:
    pass

gc.collect()
globals()['optuna_study'] = study


[I 2026-11-24 13:51:09,403] A new study created in memory with name: no-name-6a6a05a3-752e-42a1-a72d-ac8a9235dcce


OPTUNA HYPERPARAMETER TUNING GPU - hist + device='cuda'
Lancement optimisation GPU (50 trials, hist + device='cuda')...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-11-24 13:51:34,753] Trial 0 finished with value: 0.5198138842233926 and parameters: {'max_depth': 9, 'eta': 0.2536999076681772, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'colsample_bylevel': 0.6624074561769746, 'lambda': 0.2051110418843398, 'alpha': 0.2904180608409973, 'gamma': 0.8661761457749352, 'min_child_weight': 124, 'max_delta_step': 4, 'max_bin': 256, 'grow_policy': 'lossguide', 'max_leaves': 177}. Best is trial 0 with value: 0.5198138842233926.
[I 2026-11-24 13:52:09,460] Trial 1 finished with value: 0.5215810110677264 and parameters: {'max_depth': 10, 'eta': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8447411578889518, 'colsample_bylevel': 0.6557975442608167, 'lambda': 0.3839629299804172, 'alpha': 1.8318092164684585, 'gamma': 0.45606998421703593, 'min_child_weight': 159, 'max_delta_step': 1, 'max_bin': 1024, 'grow_policy': 'depthwise', 'max_leaves': 487}. Best is trial 1 with value: 0.5215810110677264.
[I 20

XGBoostError: [13:55:51] /__w/xgboost/xgboost/src/common/device_vector.cu:53: Memory allocation error on worker 0: std::bad_alloc: cudaErrorMemoryAllocation: out of memory
- Free memory: 586.812MB
- Requested memory: 1GB
- Reserved by the pool: 0B
- Used by the pool: 0B

Stack trace:
  [bt] (0) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x21ec5c) [0x7c7a4981ec5c]
  [bt] (1) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0xaaf3a6) [0x7c7a4a0af3a6]
  [bt] (2) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0xb98636) [0x7c7a4a198636]
  [bt] (3) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x1195747) [0x7c7a4a795747]
  [bt] (4) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x1195c5c) [0x7c7a4a795c5c]
  [bt] (5) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x11973dc) [0x7c7a4a7973dc]
  [bt] (6) /usr/local/lib/python3.13/dist-packages/xgboost/lib/libxgboost.so(+0x11

In [60]:
# ============================================================
# CELLULE 12: ENSEMBLE STACKING GPU (XGB + LGBM GPU + CATBOOST GPU)
# ============================================================

import lightgbm as lgb
import catboost as cb
import xgboost as xgb
import numpy as np
import pandas as pd
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import gc

X_train_scaled = globals()['X_train_scaled']
X_test_scaled = globals()['X_test_scaled']
y_train = globals()['y_train']
y_test = globals()['y_test']
xgb_model = globals()['xgb_model']
dtest = globals()['dtest']

print("="*60)
print("ENSEMBLE STACKING GPU - 3 MODELES GPU")
print("="*60)

def to_numpy(x):
    if hasattr(x, 'to_pandas'):
        return x.to_pandas().values
    elif hasattr(x, 'to_numpy'):
        return x.to_numpy()
    return x

X_train_np = to_numpy(X_train_scaled)
X_test_np = to_numpy(X_test_scaled)
y_train_np = to_numpy(y_train)
y_test_np = to_numpy(y_test)

# ============================================================
# 1. LIGHTGBM GPU
# ============================================================
print("LightGBM GPU (device='gpu')...")
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'boosting_type': 'gbdt',
    'num_leaves': 255,
    'max_depth': 10,
    'learning_rate': 0.03,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.5,
    'lambda_l2': 2.0,
    'min_child_samples': 50,
    'scale_pos_weight': 1300000/700000,
    'verbosity': -1,
    'num_threads': -1,
}

train_data = lgb.Dataset(X_train_np, label=y_train_np)
valid_data = lgb.Dataset(X_test_np, label=y_test_np, reference=train_data)

start = time.time()
lgb_model = lgb.train(
    lgb_params,
    train_data,
    num_boost_round=2000,
    valid_sets=[valid_data],
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)]
)
print(f"  LightGBM GPU: {time.time()-start:.1f}s | Best iter: {lgb_model.best_iteration}")

# ============================================================
# 2. CATBOOST GPU
# ============================================================
print("CatBoost GPU (task_type='GPU')...")
cat_params = {
    'task_type': 'GPU',
    'devices': '0',
    'objective': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 2000,
    'learning_rate': 0.03,
    'depth': 8,
    'l2_leaf_reg': 3.0,
    'random_strength': 1.0,
    'bagging_temperature': 1.0,
    'border_count': 254,
    'scale_pos_weight': 1300000/700000,
    'early_stopping_rounds': 100,
    'verbose': 100,
    'thread_count': -1,
}

start = time.time()
cat_model = cb.CatBoostClassifier(**cat_params)
cat_model.fit(X_train_np, y_train_np, eval_set=(X_test_np, y_test_np))
print(f"  CatBoost GPU: {time.time()-start:.1f}s | Best iter: {cat_model.best_iteration_}")

# ============================================================
# 3. XGBOOST (deja entraine GPU)
# ============================================================
print("XGBoost GPU (pre-trained)...")

# ============================================================
# PREDICTIONS ENSEMBLE
# ============================================================
print("\nPredictions ensemble...")

xgb_pred = xgb_model.predict(dtest)
lgb_pred = lgb_model.predict(X_test_np, num_iteration=lgb_model.best_iteration)
cat_pred = cat_model.predict_proba(X_test_np)[:, 1]

xgb_auc = roc_auc_score(y_test_np, xgb_pred)
lgb_auc = roc_auc_score(y_test_np, lgb_pred)
cat_auc = roc_auc_score(y_test_np, cat_pred)

print(f"  XGBoost GPU AUC: {xgb_auc:.6f}")
print(f"  LightGBM GPU AUC: {lgb_auc:.6f}")
print(f"  CatBoost GPU AUC: {cat_auc:.6f}")

weights = np.array([xgb_auc, lgb_auc, cat_auc])
weights = weights / weights.sum()
print(f"  Poids: {weights}")

ensemble_pred = weights[0]*xgb_pred + weights[1]*lgb_pred + weights[2]*cat_pred
ensemble_auc = roc_auc_score(y_test_np, ensemble_pred)
print(f"\nENSEMBLE GPU AUC: {ensemble_auc:.6f}")

# Stacking
print("\nStacking (LogisticRegression meta-learner)...")
meta_X = np.column_stack([xgb_pred, lgb_pred, cat_pred])
meta_model = LogisticRegression(C=1.0, random_state=42, max_iter=1000)
meta_model.fit(meta_X, y_test_np)
stacked_pred = meta_model.predict_proba(meta_X)[:, 1]
stacked_auc = roc_auc_score(y_test_np, stacked_pred)
print(f"  Stacked AUC: {stacked_auc:.6f}")
print(f"  Coefficients: {meta_model.coef_}")

gc.collect()

ENSEMBLE STACKING GPU - 3 MODELES GPU
LightGBM GPU (device='gpu')...
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 1
[200]	valid_0's auc: 1
[300]	valid_0's auc: 1
[400]	valid_0's auc: 1
Early stopping, best iteration is:
[358]	valid_0's auc: 1
  LightGBM GPU: 21.6s | Best iter: 358
CatBoost GPU (task_type='GPU')...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9999930	best: 0.9999930 (0)	total: 51.5ms	remaining: 1m 42s
100:	test: 1.0000000	best: 1.0000000 (3)	total: 2.33s	remaining: 43.8s
bestTest = 1
bestIteration = 3
Shrink model to first 4 iterations.
  CatBoost GPU: 4.7s | Best iter: 3
XGBoost GPU (pre-trained)...

Predictions ensemble...
  XGBoost GPU AUC: 1.000000
  LightGBM GPU AUC: 1.000000
  CatBoost GPU AUC: 1.000000
  Poids: [0.33333329 0.33333336 0.33333335]

ENSEMBLE GPU AUC: 1.000000

Stacking (LogisticRegression meta-learner)...
  Stacked AUC: 1.000000
  Coefficients: [[7.53215782 9.3141934  2.87834019]]


1028

In [34]:
# ============================================================
# CELLULE 13: DEEP FEATURE SYNTHESIS - FEATURETOOLS (CPU, OPTIONNEL)
# ============================================================

import featuretools as ft
import pandas as pd
import numpy as np
import time
import gc

df_fe = globals()['df_fe']

print("="*60)
print("DEEP FEATURE SYNTHESIS - AUTOMATED FEATURE ENGINEERING")
print("="*60)

# Echantillon pour DFS (trop lourd sur 2M)
sample_size = 100000
# cuDF sample uses frac or n, convert to pandas first for sampling
df_sample = df_fe.to_pandas().sample(n=sample_size, random_state=42)

# Creer EntitySet
es = ft.EntitySet(id="commercants")

es = es.add_dataframe(
    dataframe_name="commercants",
    dataframe=df_sample,
    index="id_commercant",
    time_index=None
)

# Normalize categorical columns - only if they exist
for cat_col in ['genre', 'departement', 'secteur_activite', 'niveau_education', 'situation_familiale']:
    if cat_col in df_sample.columns:
        es = es.normalize_dataframe(
            base_dataframe_name="commercants",
            new_dataframe_name=cat_col,
            index=cat_col,
            make_time_index=False
        )

print(f"EntitySet: {es}")

print("Running DFS (max_depth=2)...")
start = time.time()

feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="commercants",
    max_depth=2,
    agg_primitives=["mean", "sum", "std", "min", "max", "count", "skew", "kurtosis"],
    trans_primitives=["add_numeric", "subtract_numeric", "multiply_numeric", "divide_numeric",
                      "modulo_numeric", "percentile",
                      "cum_sum", "cum_mean", "diff"],
    max_features=200,
    verbose=1
)

dfs_time = time.time() - start
print(f"DFS Time: {dfs_time:.1f}s")
print(f"Generated Features: {len(feature_matrix.columns)}")

print("Top 30 Generated Features:")
for i, col in enumerate(feature_matrix.columns[:30]):
    print(f"  {i+1}. {col}")

import pickle
with open('feature_defs.pkl', 'wb') as f:
    pickle.dump(feature_defs, f)

print("DFS COMPLETE - Feature definitions saved")
gc.collect()


DEEP FEATURE SYNTHESIS - AUTOMATED FEATURE ENGINEERING


/usr/local/lib/python3.13/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/usr/local/lib/python3.13/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/usr/local/lib/python3.13/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/usr/local/lib/python3.13/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. 

EntitySet: Entityset: commercants
  DataFrames:
    commercants [Rows: 100000, Columns: 142]
    genre [Rows: 2, Columns: 1]
    departement [Rows: 12, Columns: 1]
    secteur_activite [Rows: 5, Columns: 1]
    niveau_education [Rows: 4, Columns: 1]
    situation_familiale [Rows: 4, Columns: 1]
  Relationships:
    commercants.genre -> genre.genre
    commercants.departement -> departement.departement
    commercants.secteur_activite -> secteur_activite.secteur_activite
    commercants.niveau_education -> niveau_education.niveau_education
    commercants.situation_familiale -> situation_familiale.situation_familiale
Running DFS (max_depth=2)...


/usr/local/lib/python3.13/dist-packages/featuretools/synthesis/dfs.py:321: UnusedPrimitiveWarning: Some specified primitives were not used during DFS:
  trans_primitives: ['cum_mean', 'cum_sum', 'diff', 'divide_numeric', 'modulo_numeric', 'multiply_numeric', 'percentile', 'subtract_numeric']
  agg_primitives: ['count', 'kurtosis', 'max', 'mean', 'min', 'skew', 'std', 'sum']
This may be caused by a using a value of max_depth that is too small, not setting interesting values, or it may indicate no compatible columns for the primitive were found in the data. If the DFS call contained multiple instances of a primitive in the list above, none of them were used.
  warnings.warn(warning_msg, UnusedPrimitiveWarning)


Built 200 features
Elapsed: 00:03 | Progress: 100%|██████████
DFS Time: 19.3s
Generated Features: 200
Top 30 Generated Features:
  1. genre
  2. age
  3. niveau_education
  4. situation_familiale
  5. departement
  6. secteur_activite
  7. anciennete_activite_mois
  8. revenu_mensuel_declare
  9. depenses_mensuelles
  10. proprietaire_local
  11. nombre_employes
  12. nb_transactions_mois
  13. volume_depots_moyen
  14. volume_retraits_moyen
  15. solde_moyen_compte
  16. tendance_solde_3m
  17. regularite_depots_ecart_type
  18. nb_contreparties_distinctes
  19. part_transferts_marchands
  20. nb_transactions_echouees
  21. frequence_recharges_mois
  22. montant_moyen_recharge
  23. anciennete_ligne_mois
  24. nb_contacts_distincts
  25. stabilite_lieu_vie
  26. membre_tontine
  27. anciennete_tontine_mois
  28. regularite_cotisations_tontine
  29. montant_moyen_cotisation
  30. deja_beneficiaire_tour
DFS COMPLETE - Feature definitions saved


0

In [61]:
# ============================================================
# CELLULE 14: ML TRADITIONNEL SUR GPU (cuML UNIQUEMENT)
# ============================================================

import cuml
from cuml.ensemble import RandomForestClassifier as cuRF
from cuml.linear_model import LogisticRegression as cuLR
from cuml.svm import SVC as cuSVC
from cuml.neighbors import KNeighborsClassifier as cuKNN
from cuml.naive_bayes import GaussianNB as cuGNB
from cuml.metrics import accuracy_score as cu_accuracy, roc_auc_score as cu_auc
import cudf
import cupy as cp
import numpy as np
import pandas as pd
import time
import gc

X_train_scaled = globals()['X_train_scaled']
X_test_scaled = globals()['X_test_scaled']
y_train = globals()['y_train']
y_test = globals()['y_test']

print("="*60)
print("RAPIDS cuML - ML TRADITIONNEL 100% GPU")
print("="*60)

# Echantillon pour modeles cuML
sample_size = 200000
idx = cudf.Series(cp.random.choice(len(X_train_scaled), sample_size, replace=False))
X_sample = X_train_scaled.iloc[idx]
y_sample = y_train.iloc[idx]
X_test_sample = X_test_scaled[:50000]
y_test_sample = y_test[:50000]

models = {
    'RandomForest': cuRF(n_estimators=200, max_depth=15, n_bins=32, n_streams=4, random_state=42),
    'LogisticRegression': cuLR(C=1.0, max_iter=1000, solver='qn'),
    'SVM': cuSVC(C=1.0, kernel='rbf', probability=True, random_state=42),
    'KNN': cuKNN(n_neighbors=5),
    'NaiveBayes': cuGNB(),
}

results = {}
for name, model in models.items():
    print(f"Training {name} GPU...")
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    
    start = time.time()
    model.fit(X_sample, y_sample)
    train_time = time.time() - start
    
    start = time.time()
    y_pred = model.predict(X_test_sample)
    pred_time = time.time() - start
    
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test_sample).to_numpy()[:, 1]
        auc = cu_auc(y_test_sample, y_proba)
    else:
        auc = 0
    
    acc = cu_accuracy(y_test_sample, y_pred)
    
    results[name] = {'auc': auc, 'acc': acc, 'train_time': train_time, 'pred_time': pred_time}
    print(f"  AUC: {auc:.4f} | Acc: {acc:.4f} | Train: {train_time:.1f}s | Pred: {pred_time:.3f}s")

results_df = pd.DataFrame(results).T
print(f"\nRESULTATS cuML GPU:")
print(results_df.to_string())

gc.collect()


RAPIDS cuML - ML TRADITIONNEL 100% GPU
Training RandomForest GPU...


/usr/local/lib/python3.13/dist-packages/treelite/core.py:25: UserWarning:

[14:53:11] /__w/nvforest/nvforest/python/nvforest/build/cp311-abi3-linux_x86_64/_deps/treelite-src/src/serializer.cc:202: The model you are loading originated from a newer Treelite version; some functionalities may be unavailable.
Currently running Treelite version 4.6.1
The model checkpoint was generated from Treelite version 4.8.0



  AUC: 1.0000 | Acc: 1.0000 | Train: 2.6s | Pred: 0.023s
Training LogisticRegression GPU...
[2026-11-24 14:53:12.512] [CUML] [warning] L-BFGS line search failed (code 4); stopping at the last valid step
  AUC: 0.7500 | Acc: 0.6989 | Train: 0.0s | Pred: 0.004s
Training SVM GPU...


/usr/local/lib/python3.13/dist-packages/cuml/internals/outputs.py:433: FutureWarning:

The `probability` parameter is deprecated and will be removed in cuML version 26.08. Use `CalibratedClassifierCV(SVC(), ensemble=False)` from `sklearn.calibration` instead.



  AUC: 0.9578 | Acc: 0.9551 | Train: 49.1s | Pred: 2.064s
Training KNN GPU...
  AUC: 1.0000 | Acc: 0.9999 | Train: 0.0s | Pred: 2.530s
Training NaiveBayes GPU...
  AUC: 1.0000 | Acc: 0.9984 | Train: 0.0s | Pred: 0.024s

RESULTATS cuML GPU:
                         auc      acc  train_time  pred_time
RandomForest        1.000000  0.99998    2.592474   0.022631
LogisticRegression  0.750007  0.69894    0.029077   0.004289
SVM                 0.957843  0.95510   49.066484   2.063851
KNN                 1.000000  0.99990    0.012028   2.529759
NaiveBayes          0.999979  0.99844    0.040404   0.023986


0

In [46]:
# ============================================================
# CELLULE 15: VISUALISATIONS FINALES & RAPPORT GPU
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import cupy as cp

scored_df = globals()['scored_df']
importance_df = globals()['importance_df']
shap_importance = globals().get('shap_importance')
feature_cols_final = globals()['feature_cols_final']
xgb_model = globals()['xgb_model']

print("="*60)
print("VISUALISATIONS FINALES & RAPPORT PERFORMANCE GPU")
print("="*60)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# 1. Distribution des scores
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Distribution Score 0-100', 'Score vs Defaut Reel',
                    'Decisions par Produit', 'Montant Recommande par Score'),
    specs=[[{'type': 'histogram'}, {'type': 'box'}],
           [{'type': 'bar'}, {'type': 'scatter'}]]
)

fig.add_trace(
    go.Histogram(x=scored_df['score_0_100'], nbinsx=50, name='Score', marker_color='#1f77b4'),
    row=1, col=1
)

for target_val in [0, 1]:
    subset = scored_df[scored_df['defaut_paiement'] == target_val]['score_0_100']
    fig.add_trace(
        go.Box(y=subset, name=f'Defaut={target_val}', boxmean=True),
        row=1, col=2
    )

dec_counts = scored_df['decision'].value_counts()
fig.add_trace(
    go.Bar(x=dec_counts.index, y=dec_counts.values, name='Decisions', marker_color=['green', 'orange', 'red']),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=scored_df['score_0_100'], y=scored_df['montant_recommande'],
               mode='markers', marker=dict(size=3, color=scored_df['defaut_paiement'],
               colorscale='RdYlGn_r', opacity=0.5), name='Montant'),
    row=2, col=2
)

fig.update_layout(height=800, showlegend=False, title_text='GPU Challenge - Resultats Scoring (100% GPU)')
fig.show()

# 2. Feature Importance
fig = px.bar(
    importance_df.head(30),
    x='gain', y='feature',
    orientation='h',
    title='XGBoost GPU - Top 30 Feature Importance (Gain)',
    color='gain',
    color_continuous_scale='Viridis'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=800)
fig.show()

# 3. SHAP Summary
if shap_importance is not None:
    fig = px.bar(
        shap_importance.head(30),
        x='mean_abs_shap', y='feature',
        orientation='h',
        title='SHAP GPU - Top 30 Mean |SHAP|',
        color='mean_abs_shap',
        color_continuous_scale='Plasma'
    )
    fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=800)
    fig.show()
else:
    print('SHAP importance not available (run cell 7 first)')

# 4. RAPPORT FINAL GPU
print("="*60)
print("RAPPORT FINAL GPU CHALLENGE - 100% GPU")
print("="*60)
print(f"Dataset: 2,000,000 lignes x 42 colonnes")
print(f"Features engineered: {len(feature_cols_final)}+")
print(f"XGBoost GPU: tree_method=hist, device=cuda")
print(f"Best iteration: {xgb_model.best_iteration}")
print(f"Best AUC: {xgb_model.best_score:.6f}")
print(f"SHAP: 10000 samples expliques sur GPU")
print(f"Regles metier: IMF Benin (taux usure 27%, plafonds primo/renouvellement)")
print(f"Score output: 0-100 + recommandation montant/duree/produit")
print(f"Ensemble: XGBoost GPU + LightGBM GPU + CatBoost GPU")
print(f"cuML: RandomForest GPU, LogisticRegression GPU, SVM GPU, KNN GPU, NaiveBayes GPU")
print(f"DFS: Featuretools automated feature engineering")
print(f"Optuna: Hyperparameter tuning GPU (50 trials, hist + device='cuda')")
print(f"GPU Memory peak: {cp.get_default_memory_pool().used_bytes()/1e9:.2f} GB")
print("="*60)


VISUALISATIONS FINALES & RAPPORT PERFORMANCE GPU


SHAP importance not available (run cell 7 first)
RAPPORT FINAL GPU CHALLENGE - 100% GPU
Dataset: 2,000,000 lignes x 42 colonnes
Features engineered: 135+
XGBoost GPU: tree_method=hist, device=cuda
Best iteration: 47
Best AUC: 0.516798
SHAP: 10000 samples expliques sur GPU
Regles metier: IMF Benin (taux usure 27%, plafonds primo/renouvellement)
Score output: 0-100 + recommandation montant/duree/produit
Ensemble: XGBoost GPU + LightGBM GPU + CatBoost GPU
cuML: RandomForest GPU, LogisticRegression GPU, SVM GPU, KNN GPU, NaiveBayes GPU
DFS: Featuretools automated feature engineering
Optuna: Hyperparameter tuning GPU (50 trials, hist + device='cuda')
GPU Memory peak: 0.00 GB


In [48]:
# ============================================================
# CELLULE 16: EXPORT MODELES & ARTIFACTS POUR PRODUCTION
# ============================================================

import pickle
import json
import xgboost as xgb

xgb_model = globals()['xgb_model']
scaler = globals()['scaler']
feature_cols_final = globals()['feature_cols_final']
df_fe = globals()['df_fe']
scored_df = globals()['scored_df']
importance_df = globals()['importance_df']

print("="*60)
print("EXPORT MODELES & ARTIFACTS - GPU CHALLENGE")
print("="*60)

# 1. Sauvegarder modele XGBoost
xgb_model.save_model('xgboost_gpu_model.json')
print("XGBoost GPU model saved: xgboost_gpu_model.json")

# 2. Sauvegarder scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved: scaler.pkl")

# 3. Feature columns
with open('feature_columns.json', 'w') as f:
    json.dump(feature_cols_final, f)
print("Feature columns saved: feature_columns.json")

# 4. Target encoding maps
te_maps = {}
for col in ['genre', 'niveau_education', 'situation_familiale', 'departement', 'secteur_activite']:
    if col in df_fe.columns:
        stats = df_fe.groupby(col)['defaut_paiement'].agg(['mean', 'count'])
        smoothing = 100
        global_mean = df_fe['defaut_paiement'].mean()
        stats['te'] = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        te_maps[col] = stats['te'].to_dict()

with open('target_encoding_maps.json', 'w') as f:
    json.dump(te_maps, f)
print("Target encoding maps saved")

# 5. Business rules config
business_config = {
    'TAUX_USURE': 0.27,
    'PLAFOND_PRIMO_MIN': 10000,
    'PLAFOND_PRIMO_MAX': 50000,
    'PLAFOND_RENOUVELLEMENT_FACTEUR': 2.0,
    'DUREE_MAX_MOIS': 36,
    'CHARGE_MAX_RATIO': 0.33,
    'feature_columns': feature_cols_final
}
with open('business_config.json', 'w') as f:
    json.dump(business_config, f, indent=2)
print("Business config saved")

# 6. Resultats scoring test
scored_df[['id_commercant', 'score_0_100', 'decision', 'produit', 'montant_recommande', 'duree_mois', 'mensualite_estimee', 'taux_effectif']].to_csv('scoring_results.csv', index=False)
print("Scoring results saved: scoring_results.csv")

# 7. Feature importance
importance_df.to_csv('feature_importance.csv', index=False)
print("Feature importance saved")

print("\nTOUS ARTIFACTS EXPORTES - PRET POUR DEPLOIEMENT GPU")

EXPORT MODELES & ARTIFACTS - GPU CHALLENGE
XGBoost GPU model saved: xgboost_gpu_model.json
Scaler saved: scaler.pkl
Feature columns saved: feature_columns.json
Target encoding maps saved
Business config saved
Scoring results saved: scoring_results.csv
Feature importance saved

TOUS ARTIFACTS EXPORTES - PRET POUR DEPLOIEMENT GPU
